In [21]:
# CELL 1: Imports
import librosa
import numpy as np
import xgboost as xgb
import pickle
import tensorflow as tf
import tensorflow_hub as hub
import warnings

# Suppress warnings for cleaner output
# warnings.filterwarnings('ignore')

In [24]:
# CELL 2: Load Models and Dependencies
import pickle
import tensorflow_hub as hub
import xgboost as xgb

# 1. Load PerchV2
PERCH_MODEL_URL = "https://www.kaggle.com/models/google/bird-vocalization-classifier/TensorFlow2/bird-vocalization-classifier/4"
print("Loading PerchV2 model...")
perch_model = hub.load(PERCH_MODEL_URL)

# 2. Define XGBoost Framework and Load Parameters
print("Initializing XGBoost framework and loading parameters...")
NUM_CLASSES = 234

# Pasting the original framework from train.py
xgb_clf = xgb.XGBClassifier(
    objective='multi:softprob', 
    num_class=NUM_CLASSES,      
    n_estimators=300,
    max_depth=6,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    tree_method='hist',  
    device='cuda', # Note: Change to 'cpu' if running inference without a GPU
    random_state=42
)

# Load the trained weights into the framework
xgb_clf.load_model(r'C:\Users\DuyThanh\Desktop\Learn4ever\Python\FraudDetection\birdCLEF\BirdCLEF2026\parameters.json') 

# 3. Load the Two Encoders
print("Loading label encoders...")
# First PKL: The scikit-learn LabelEncoder from training
with open(r'C:\Users\DuyThanh\Desktop\Learn4ever\Python\FraudDetection\birdCLEF\BirdCLEF2026\label_encoder.pkl', 'rb') as f:
    temp_encoder = pickle.load(f)

# Second PKL: Your converted label2idx dictionary
with open(r'C:\Users\DuyThanh\Desktop\Learn4ever\Python\FraudDetection\birdCLEF\BirdCLEF2026\label2idx.pkl', 'rb') as f:
    label2idx = pickle.load(f)


# Create a master list of all 234 string labels we need for the final output
ALL_SPECIES = list(label2idx.keys())
    
print("All models and encoders loaded successfully!")

Loading PerchV2 model...
Initializing XGBoost framework and loading parameters...
Loading label encoders...
All models and encoders loaded successfully!


In [25]:
# CELL 3: Audio Preprocessing Function

def preprocess_audio(file_path, target_sr=32000, duration=5.0):
    """
    Loads an audio file, resamples it to 32kHz, and limits to 5 seconds.
    Pads with zeros if the audio is shorter than 5 seconds.
    """
    # Load audio (librosa automatically resamples if sr is provided)
    audio, sr = librosa.load(file_path, sr=target_sr, duration=duration)
    
    # Calculate target length in samples
    target_length = int(target_sr * duration)
    
    # Pad if shorter than 5 seconds
    if len(audio) < target_length:
        padding = target_length - len(audio)
        audio = np.pad(audio, (0, padding), mode='constant')
        
    return audio

In [26]:
# CELL 4: Embedding Extraction Function

def get_perch_embeddings(audio_array):
    """
    Passes the audio array through PerchV2 to get the feature vector.
    Safely handles both tuple and dictionary model outputs.
    """
    # Perch expects a batch dimension and float32 type
    audio_batch = np.float32(audio_array[np.newaxis, :])
    
    # Run inference
    infer_results = perch_model.infer_tf(audio_batch)
    
    # EXTRACT EMBEDDINGS SAFELY
    if isinstance(infer_results, tuple):
        # Perch infer_tf usually returns a tuple: (logits, embeddings)
        # Logits have ~10,000+ dimensions (species), embeddings have 1280 or 1536 dimensions.
        for item in infer_results:
            if item.shape[-1] < 2000:  # Isolates the embedding vector
                embeddings = item.numpy()
                break
                
    elif isinstance(infer_results, dict):
        # If the model returns a dict, grab it by the key
        embeddings = infer_results.get('embeddings', infer_results.get('output_1')).numpy()
        
    else:
        # Fallback if the model directly returns a single tensor
        embeddings = infer_results.numpy()
        
    return embeddings

In [27]:
# CELL 5: Prediction Function (Probabilities)

def predict_with_xgboost(embedding_vector):
    """
    Takes the embedding vector and returns the softmax probability array.
    """
    if embedding_vector.ndim == 1:
        embedding_vector = embedding_vector.reshape(1, -1)
    
    # .predict_proba() returns the softmax probabilities
    probabilities = xgb_clf.predict_proba(embedding_vector)[0]
    
    return probabilities

In [28]:
# CELL 6: Decoding and Padding Function

def decode_and_pad_probabilities(xgb_probs, padding_value=0.0001):
    """
    Maps the XGBoost probabilities to their string names.
    Fills in missing classes with a padding_value to avoid absolute 0%.
    """
    # Step 1: Initialize the final dictionary with the padding value for ALL 234 species
    final_probabilities = {species: padding_value for species in ALL_SPECIES}
    
    # Step 2: Overwrite the padded values with the actual predictions
    for idx, prob in enumerate(xgb_probs):
        # Get the actual string name from your Scikit-Learn encoder
        species_name = temp_encoder.inverse_transform([idx])[0]
        
        # Update the dictionary, ensuring we never drop below the padding value
        final_probabilities[species_name] = max(float(prob), padding_value)
        
    return final_probabilities

In [33]:
# CELL 7: Run the Full Pipeline and Generate Submission
import os
import pandas as pd

# The baseline Kaggle padding value (1 / 234 classes)
KAGGLE_PAD_VALUE = 1.0 / len(ALL_SPECIES)

def process_and_format_audio(file_path):
    print(f"Processing: {file_path}")
    


    row_id = os.path.splitext(os.path.basename(file_path))[0]
    
    # 2. Run the core pipeline steps
    audio = preprocess_audio(file_path)
    embedding = get_perch_embeddings(audio)
    raw_probs = predict_with_xgboost(embedding)
    
    # 3. Decode with the exact 1/234 padding value
    final_results = decode_and_pad_probabilities(raw_probs, padding_value=KAGGLE_PAD_VALUE)
    
    # 4. Structure the final dictionary for Pandas
    # We put row_id first, then alphabetically sort the species to match Kaggle's standard format
    formatted_row = {'row_id': row_id}
    
    for species in sorted(ALL_SPECIES):
        formatted_row[species] = final_results[species]
        
    return formatted_row


In [ ]:
# --- TEST IT AND EXPORT TO CSV ---
# You can put multiple test files in this list
test_files = [r'C:\Users\DuyThanh\Desktop\Learn4ever\Python\FraudDetection\birdCLEF\BirdCLEF2026\iNat1460166.ogg'] 
all_submission_rows = []

for file_path in test_files:
    try:
        row_data = process_and_format_audio(file_path)
        all_submission_rows.append(row_data)
    except Exception as e:
        print(f"An error occurred processing {file_path}: {e}")

# Create the final Kaggle DataFrame
submission_df = pd.DataFrame(all_submission_rows)

# Display the dataframe so you can verify it looks exactly like your image
display(submission_df.head())

# Save it to the standard Kaggle submission file
# submission_df.to_csv('submission.csv', index=False)
# print("\nSuccess! Saved to submission.csv")

Processing: C:\Users\DuyThanh\Desktop\Learn4ever\Python\FraudDetection\birdCLEF\BirdCLEF2026\iNat1460166.ogg


,row_id,1161364,116570,1176823,1491113,1595929,209233,22930,22956,22961,...,whnjay1,whtdov,whwpic1,y00678,yebcar,yebela1,yecmac,yecpar,yehcar1,yeofly1
0,iNat1460166,0.004274,0.149676,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,...,0.004274,0.004526,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.006435,0.004274



Success! Saved to submission.csv
